> Experiments with `xsa` module

In [23]:
import pathlib
import os
import time

import pandas as pd

import trapallada.xsa

from astroquery.esa.xmm_newton import XMMNewton
# import astropy.io
from astropy.table import Table

import altair as alt

So that the notebook is aware of modifications in the the `import`ed modules

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


All the available tables (only a few are printed)

In [3]:
tables = trapallada.xsa.get_all_tables()
tables[:5]

INFO: Retrieving tables... [astroquery.utils.tap.core]
INFO: Parsing tables... [astroquery.utils.tap.core]
INFO: Done. [astroquery.utils.tap.core]


['tap_config.coord_sys',
 'tap_config.properties',
 'tap_schema.columns',
 'tap_schema.key_columns',
 'tap_schema.keys']

Get *metadata* for observations whose duration is in a given `duration_interval`

In [4]:
obs_metadata = trapallada.xsa.get_observations_in_interval(duration_interval=(35_000, 45_000))
obs_metadata

number of chunks: 1


,observation_id,duration
0,0006010301,35210
1,0006010401,35915
2,0008220201,42610
3,0009650101,41415
4,0009650201,36590
...,...,...
1414,0810881501,35000
1415,0842811031,39981
1416,0842812034,36421
1417,0861950231,38363


The directory where observations' data will be saved

In [5]:
output_directory =  pathlib.Path(os.environ['MY_DATA_DIR']) / 'trapallada'
output_directory.mkdir(exist_ok=True)

Let us select a few observations to download

In [6]:
obs_to_download = obs_metadata['observation_id'].iloc[:10]
obs_to_download

0    0006010301
1    0006010401
2    0008220201
3    0009650101
4    0009650201
5    0011420101
6    0011420201
7    0011430201
8    0011450201
9    0011830201
Name: observation_id, dtype: object

We download the first few observations

In [7]:
for obs_id in obs_to_download:

    print(f'Requesting observation {obs_id}')
    
    trapallada.xsa.download_extract_filter(obs=obs_id, output_directory=output_directory)
    
    # in order to, hopefully, avoid bans/errors...
    time.sleep(5)

Requesting observation 0006010301
found directory "/home/manu/data/trapallada/0006010301", skipping...
Requesting observation 0006010401
found directory "/home/manu/data/trapallada/0006010401", skipping...
Requesting observation 0008220201
found directory "/home/manu/data/trapallada/0008220201", skipping...
Requesting observation 0009650101
found directory "/home/manu/data/trapallada/0009650101", skipping...
Requesting observation 0009650201
found directory "/home/manu/data/trapallada/0009650201", skipping...
Requesting observation 0011420101
found directory "/home/manu/data/trapallada/0011420101", skipping...
Requesting observation 0011420201
found directory "/home/manu/data/trapallada/0011420201", skipping...
Requesting observation 0011430201
found directory "/home/manu/data/trapallada/0011430201", skipping...
Requesting observation 0011450201
found directory "/home/manu/data/trapallada/0011450201", skipping...
Requesting observation 0011830201
found directory "/home/manu/data/trapal

In [8]:
# trapallada.xsa.download_extract_filter(obs='0011420101', output_directory=output_directory)
# trapallada.xsa.download_extract_filter(obs='0011420201', output_directory=output_directory)

## Data for Javi's paper

In [9]:
obs = '0605950101' # Javi's paper
obs_dir = trapallada.xsa.download_extract_filter(obs=obs, output_directory=output_directory)
# obs_dir = trapallada.xsa.download_extract_filter(obs=obs, output_directory=output_directory, keep_tar=True)
obs_dir

found directory "/home/manu/data/trapallada/0605950101", skipping...


PosixPath('/home/manu/data/trapallada/0605950101')

The downloaded files

In [10]:
# list(obs_dir.iterdir())
fits_files = list(obs_dir.glob('*.FTZ'))
fits_files[:3]

[PosixPath('/home/manu/data/trapallada/0605950101/P0605950101PNS003SRSPEC000C.FTZ'),
 PosixPath('/home/manu/data/trapallada/0605950101/P0605950101PNS003SRCTSR802C.FTZ'),
 PosixPath('/home/manu/data/trapallada/0605950101/P0605950101PNS003SRCTSR8016.FTZ')]

In [11]:
sample_file = obs_dir / 'P0605950101PNX000SRCTSR8001.FTZ'
assert sample_file.exists()

The file is open, the list of *Header Data Unit*s (HDUs) inside shown, and the file is closed.

In [12]:
# hdul = astropy.io.fits.open(sample_file)
# hdul.info()
# hdul.close()

In [13]:
df = trapallada.xsa.read_fits_file(sample_file)
df.head(10)

,TIME,RATE,ERROR,FRACEXP,BACKV,BACKE,RATE1,RATE1_ERR,BACK1V,BACK1E,...,RATE4_ERR,BACK4V,BACK4E,RATE5,RATE5_ERR,BACK5V,BACK5E,EXP_ID,TIMEDEL,date
0,3.604038e+08,0.878811,0.878811,0.779384,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,b'S003',1.46,1981-06-03 08:10:36.517454016
1,3.604038e+08,0.785685,0.785685,0.871763,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.785685,0.000000,0.000000,0.0,0.0,0.0,0.0,b'S003',1.46,1981-06-03 08:10:37.977454016
2,3.604038e+08,0.000000,0.000000,0.857033,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,b'S003',1.46,1981-06-03 08:10:39.437454016
3,3.604038e+08,0.000000,0.000000,0.878588,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,b'S003',1.46,1981-06-03 08:10:40.897454016
4,3.604038e+08,0.000000,0.000000,0.874164,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,b'S003',1.46,1981-06-03 08:10:42.357454016
5,3.604038e+08,2.033699,1.438042,0.673582,0.000000,0.000000,1.016849,1.016849,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,b'S003',1.46,1981-06-03 08:10:43.817454016
6,3.604038e+08,0.815694,0.815694,0.839691,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,b'S003',1.46,1981-06-03 08:10:45.277454016
7,3.604038e+08,0.795970,0.795970,0.860499,0.000000,0.000000,0.795970,0.795970,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,b'S003',1.46,1981-06-03 08:10:46.737454080
8,3.604038e+08,1.563630,1.105653,0.876079,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,b'S003',1.46,1981-06-03 08:10:48.197454016
9,3.604038e+08,-0.199282,0.127301,0.857555,0.199303,0.140928,-0.099265,0.099265,0.099265,0.099265,...,0.099265,0.099265,0.099265,0.0,0.0,0.0,0.0,b'S003',1.46,1981-06-03 08:10:49.657454016


Sampling period can be found as

In [14]:
df['date'][1] - df['date'][0]

Timedelta('0 days 00:00:01.460000')

A *rolling window* is used to bin collapse together different measurements. The *label* (within the index) of every window is that of the element right **after** the window finishes.

In [15]:
window_size = 63
ts = df.set_index('date')['RATE'].rolling(window=window_size, center=False, closed='left', step=window_size).sum()
ts

date
1981-06-03 08:10:36.517454016          NaN
1981-06-03 08:12:08.497454016    35.652579
1981-06-03 08:13:40.477454016    27.559837
1981-06-03 08:15:12.457454016    49.578663
1981-06-03 08:16:44.437454016    38.597875
                                   ...    
1981-06-03 20:57:06.517454016    25.611620
1981-06-03 20:58:38.497454016    28.518611
1981-06-03 21:00:10.477454080    28.879920
1981-06-03 21:01:42.457454016    33.363138
1981-06-03 21:03:14.437454016    30.898983
Name: RATE, Length: 505, dtype: float64

In [21]:
rate_series = trapallada.xsa.time_series_from_df(df)
rate_series

date
1981-06-03 08:12:08.497454016    34.577186
1981-06-03 08:13:40.477454016    28.529714
1981-06-03 08:15:12.457454016    48.805367
1981-06-03 08:16:44.437454016    39.296789
1981-06-03 08:18:16.417454016    36.757868
                                   ...    
1981-06-03 20:57:06.517454016    22.408824
1981-06-03 20:58:38.497454016    29.378926
1981-06-03 21:00:10.477454080    29.943388
1981-06-03 21:01:42.457454016    32.310325
1981-06-03 21:03:14.437454016    30.126159
Name: RATE, Length: 504, dtype: float64

In [24]:
alt.Chart(ts.to_frame().reset_index()).mark_line().encode(
    x='date:T',
    y='RATE:Q'
).properties(width=1_200)

alt.Chart(...)

In [17]:
# df.iloc[:3]

In [18]:
# df.iloc[:3].set_index('date')['RATE'].rolling(window=2, center=False, step=2).sum()

# Low-level calls to `XMMNewton`

## v_instrument_mode

*Instrument mode* table

In [19]:
instrument_mode_table = 'v_instrument_mode'
XMMNewton.get_columns('xsa.' + instrument_mode_table)

INFO: Retrieving tables... [astroquery.utils.tap.core]
INFO: Parsing tables... [astroquery.utils.tap.core]
INFO: Done. [astroquery.utils.tap.core]


['instrument', 'instrument_mode_oid', 'mode', 'mode_friendly_name']

A sample

In [20]:
table = XMMNewton.query_xsa_tap(
    f'select TOP 10 * from {instrument_mode_table}',
)
table.to_pandas()

,instrument,instrument_mode_oid,mode,mode_friendly_name
0,EMOS1,27,CcdDiagnostic,Diagnostic 1x1
1,EMOS1,28,Diagnostic1x1,Diagnostic 1x1
2,EMOS1,29,Diagnostic3x3,Diagnostic 3x3
3,EMOS1,30,FastCompressed,Timing Compressed
4,EMOS1,31,FastUncompressed,Timing
5,EMOS1,32,OffsetVariance,Offset Var.
6,EMOS1,33,PrimeFullWindow,Full Frame
7,EMOS1,34,PrimePartialRFS,Refresh Frame Store
8,EMOS1,35,PrimePartialW2,Small Window
9,EMOS1,36,PrimePartialW3,Large Window
